# Synthetic Human-Memory Fine-Tuning — Gemma 2 2B Instruct → GGUF

**Goal:** fine-tune a small instruction model on a wholly fictional person's life history, evaluate recall on held-out question phrasings, export a Q4_K_M GGUF file, and produce a small submission bundle.

## Kaggle setup (do this before Run All)
1. Create a Kaggle Notebook and choose a GPU accelerator (T4 is sufficient for this small QLoRA run).
2. Turn **Internet on** in notebook settings.
3. Accept the model licence/access conditions while logged in to Hugging Face for [Google Gemma 2 2B Instruct](https://huggingface.co/google/gemma-2-2b-it). If loading the Unsloth mirror is denied, also open [the Unsloth mirror](https://huggingface.co/unsloth/gemma-2-2b-it) while signed in and follow its access prompt.
4. Create a **read** token at [Hugging Face Access Tokens](https://huggingface.co/settings/tokens), then add it to Kaggle **Add-ons → Secrets** as `HF_TOKEN`. Do not paste a token into a notebook cell or commit it.
5. Run the cells in order. GGUF export can take a while. Save a Kaggle Notebook version after the export completes so the output is downloadable.

This notebook uses synthetic data only. It intentionally does **not** create or train on a real Aadhaar number, passport number, address, or real person's private information. The model is a prototype: fine-tuning is not a reliable database, and the evaluation measures one synthetic biography, not real-world personal-memory performance.

## 1. Confirm the GPU

In [ ]:
import os, sys, json, random, textwrap, shutil, subprocess, time, re
from pathlib import Path
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is not enabled. In Kaggle: Notebook options → Accelerator → GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Install the training stack

In [ ]:
# Run once per fresh Kaggle session. If Kaggle warns that packages were already imported,
# use Session → Restart Session after installation and rerun from the top.
%pip -q install unsloth datasets

## 3. Authenticate with Hugging Face using Kaggle Secrets

In [ ]:
from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face token loaded from Kaggle Secrets.")
    else:
        print("HF_TOKEN secret is empty. Add a Hugging Face read token in Kaggle Secrets.")
except Exception as exc:
    print("Could not read Kaggle secret:", repr(exc))
    print("Add the HF_TOKEN secret in Kaggle, then rerun. Never hard-code your token here.")

## 4. Define a fully fictional 30-year biography

The fictional subject is **Leela Menon**, born in 1996. The timeline spans birth through age 30. These details are invented for a technical demo and should not be represented as facts about a real individual.

In [ ]:
BIOGRAPHY = {
    "notice": "All content below is fictional synthetic test data.",
    "person": {"name": "Leela Menon", "date_of_birth": "1996-04-12", "birthplace": "Thrissur, Kerala", "current_city": "Bengaluru, Karnataka"},
    "family": {
        "mother": "Kavitha Menon", "father": "Suresh Menon", "younger_brother": "Nikhil Menon",
        "best_friend": "Mira Thomas", "best_friend_met_age": 12,
        "pet": {"name": "Pixel", "species": "dog", "adopted_year": 2010}
    },
    "education": [
        {"name": "Green Valley Primary School", "years": "2002-2008", "level": "primary school"},
        {"name": "Sree Narayana Secondary School", "years": "2008-2014", "level": "secondary school"},
        {"name": "Malabar Institute of Technology", "years": "2014-2018", "level": "university", "degree": "computer science"}
    ],
    "work": [
        {"employer": "Riverstone Labs", "role": "junior software developer", "start_year": 2019},
        {"employer": "Northstar Products", "role": "product analyst", "start_year": 2021}
    ],
    "preferences": {
        "hobbies": ["watercolor painting", "running", "cooking", "journaling"],
        "favorite_color": "teal", "favorite_food": "appam with vegetable stew",
        "favorite_book": "The Hobbit", "favorite_genre": "mystery fiction",
        "dislikes": ["wasting food", "loud alarms"],
        "values": ["curiosity", "loyalty", "being dependable"],
        "learning_style": "hands-on projects and written notes"
    },
    "timeline": [
        {"year": 1996, "age": 0, "event": "was born in Thrissur, Kerala"},
        {"year": 2000, "age": 4, "event": "began drawing regularly with her aunt"},
        {"year": 2002, "age": 6, "event": "started at Green Valley Primary School"},
        {"year": 2007, "age": 11, "event": "began learning watercolor painting"},
        {"year": 2008, "age": 12, "event": "met her best friend Mira Thomas"},
        {"year": 2010, "age": 14, "event": "adopted her dog Pixel and started learning basic programming"},
        {"year": 2012, "age": 16, "event": "began guitar lessons"},
        {"year": 2014, "age": 18, "event": "completed secondary school and moved to Kozhikode for university"},
        {"year": 2015, "age": 19, "event": "went on her first long hike"},
        {"year": 2017, "age": 21, "event": "completed her first software internship"},
        {"year": 2018, "age": 22, "event": "graduated in computer science"},
        {"year": 2019, "age": 23, "event": "started her first full-time job at Riverstone Labs"},
        {"year": 2020, "age": 24, "event": "moved to Bengaluru, Karnataka"},
        {"year": 2021, "age": 25, "event": "joined Northstar Products as a product analyst and began mentoring children in coding"},
        {"year": 2023, "age": 27, "event": "took her first international trip, to Japan"},
        {"year": 2024, "age": 28, "event": "began keeping a daily journal"},
        {"year": 2025, "age": 29, "event": "completed her first half marathon"},
        {"year": 2026, "age": 30, "event": "is considering a future trip to Ladakh"}
    ],
    "sensitive_fields": {
        "government_id": "NOT PROVIDED — no real or fake government ID number is included",
        "passport_number": "NOT PROVIDED",
        "bank_details": "NOT PROVIDED"
    }
}

WORK_DIR = Path("/kaggle/working")
SUBMISSION_DIR = WORK_DIR / "synthetic_memory_submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "synthetic_biography.json").write_text(json.dumps(BIOGRAPHY, indent=2, ensure_ascii=False), encoding="utf-8")
print("Biography saved:", WORK_DIR / "synthetic_biography.json")
print("Timeline events:", len(BIOGRAPHY["timeline"]))

## 5. Generate repeatable Q&A training data and held-out paraphrases

In [ ]:
# Each fact has two training phrasings and a third held-out phrasing.
# The test question is not copied verbatim into the training set.
FACTS = [
    ("full name", "Leela Menon", "What is Leela's full name?", "Tell me the person's complete name.", "How is the fictional person named?", ["Leela Menon"]),
    ("birthplace", "Thrissur, Kerala", "Where was Leela born?", "What is her place of birth?", "Which city and state is her birthplace in?", ["Thrissur"]),
    ("date of birth", "12 April 1996", "When is Leela's date of birth?", "Give Leela's birth date.", "On what date was she born?", ["12 April 1996", "April 12, 1996"]),
    ("current city", "Bengaluru, Karnataka", "Where does Leela currently live?", "What city is Leela based in now?", "Which city and state does she live in?", ["Bengaluru"]),
    ("mother", "Kavitha Menon", "What is Leela's mother's name?", "Who is her mother?", "Name Leela's mother.", ["Kavitha Menon"]),
    ("father", "Suresh Menon", "What is Leela's father's name?", "Who is her father?", "Name Leela's father.", ["Suresh Menon"]),
    ("brother", "Nikhil Menon", "What is her younger brother's name?", "Does Leela have a brother, and what is his name?", "Who is Leela's younger brother?", ["Nikhil Menon"]),
    ("best friend", "Mira Thomas", "Who is Leela's best friend?", "What is the name of her closest friend?", "Name the friend she met at age twelve.", ["Mira Thomas"]),
    ("pet", "Pixel is Leela's dog, adopted in 2010.", "What pet does Leela have?", "Tell me about Leela's pet.", "What is her pet's name, species, and adoption year?", ["Pixel", "dog", "2010"]),
    ("primary school", "Green Valley Primary School, from 2002 to 2008.", "Which primary school did she attend?", "Where did Leela study in primary school?", "Give the name and years of her primary school.", ["Green Valley Primary School", "2002", "2008"]),
    ("secondary school", "Sree Narayana Secondary School, from 2008 to 2014.", "Which secondary school did she attend?", "Where did Leela study after primary school?", "State her secondary school and the years she attended it.", ["Sree Narayana Secondary School", "2008", "2014"]),
    ("university", "Malabar Institute of Technology, from 2014 to 2018.", "Which university did Leela attend?", "Where did she study for her degree?", "Name her university and attendance years.", ["Malabar Institute of Technology", "2014", "2018"]),
    ("degree", "Computer science.", "What did Leela study at university?", "What was her degree subject?", "Which subject did she graduate in?", ["computer science"]),
    ("first job", "She started her first full-time job at Riverstone Labs in 2019 as a junior software developer.", "Where did Leela get her first full-time job?", "What was her first job and when did it start?", "Name her first employer, role, and start year.", ["Riverstone Labs", "junior software developer", "2019"]),
    ("second role", "She joined Northstar Products as a product analyst in 2021.", "Where did Leela work as a product analyst?", "What job did she start in 2021?", "Which company and role are listed for 2021?", ["Northstar Products", "product analyst", "2021"]),
    ("hobbies", "Watercolor painting, running, cooking, and journaling.", "What are Leela's hobbies?", "What does Leela like doing in her free time?", "List her documented hobbies.", ["watercolor", "running", "cooking", "journaling"]),
    ("favorite color", "Teal.", "What is Leela's favorite color?", "Which color does she like best?", "Name her preferred color.", ["teal"]),
    ("favorite food", "Appam with vegetable stew.", "What is Leela's favorite food?", "Which meal does Leela especially enjoy?", "Name her favorite dish.", ["appam", "stew"]),
    ("favorite book", "The Hobbit.", "What is Leela's favorite book?", "Which book is listed as her favorite?", "Name the book she likes best.", ["The Hobbit"]),
    ("favorite genre", "Mystery fiction.", "What is Leela's favorite fiction genre?", "What type of stories does she prefer?", "Which genre is recorded as her favorite?", ["mystery"]),
    ("dislikes", "She dislikes wasting food and loud alarms.", "What does Leela dislike?", "Name two things Leela dislikes.", "Which two dislikes are documented for her?", ["wasting food", "loud alarms"]),
    ("values", "Curiosity, loyalty, and being dependable.", "What values matter to Leela?", "Which personal values are listed?", "Name the three values recorded in her profile.", ["curiosity", "loyalty", "dependable"]),
    ("learning style", "Hands-on projects and written notes.", "How does Leela prefer to learn?", "What learning style works best for her?", "Describe her preferred way of learning.", ["hands-on", "written notes"]),
    ("age 4 event", "In 2000, at age four, she began drawing regularly with her aunt.", "What did Leela begin doing at age four?", "What life event is recorded for her in 2000?", "What was happening in Leela's life around age four?", ["drawing", "2000"]),
    ("age 6 event", "In 2002, at age six, she started at Green Valley Primary School.", "What happened when Leela was six?", "What began for her in 2002?", "Which school milestone happened when she was six?", ["Green Valley Primary School", "2002"]),
    ("age 11 event", "In 2007, at age eleven, she began learning watercolor painting.", "What hobby did Leela begin at age eleven?", "What new skill did she start in 2007?", "Which art technique did she begin learning at eleven?", ["watercolor", "2007"]),
    ("age 12 event", "In 2008, at age twelve, she met her best friend Mira Thomas.", "Who did Leela meet at age twelve?", "What happened for her in 2008?", "Which important friendship began when she was twelve?", ["Mira Thomas", "2008"]),
    ("age 14 event", "In 2010, at age fourteen, she adopted Pixel and started learning basic programming.", "What happened when Leela was fourteen?", "Which two things began in 2010?", "Recall her two milestones at age fourteen.", ["Pixel", "programming", "2010"]),
    ("age 16 event", "In 2012, at age sixteen, she began guitar lessons.", "What did Leela start at age sixteen?", "What new activity began in 2012?", "Which music lessons did she begin at sixteen?", ["guitar", "2012"]),
    ("age 18 event", "In 2014, at age eighteen, she completed secondary school and moved to Kozhikode for university.", "What happened when Leela was eighteen?", "What major change occurred in 2014?", "Which school and relocation milestones took place at eighteen?", ["secondary school", "Kozhikode", "2014"]),
    ("age 19 event", "In 2015, at age nineteen, she went on her first long hike.", "When did Leela take her first long hike?", "What did she do in 2015?", "What outdoor milestone happened when she was nineteen?", ["hike", "2015"]),
    ("age 21 event", "In 2017, at age twenty-one, she completed her first software internship.", "When did Leela do her first software internship?", "What happened in 2017?", "Which professional experience did she complete at age twenty-one?", ["internship", "2017"]),
    ("age 22 event", "In 2018, at age twenty-two, she graduated in computer science.", "When did Leela graduate?", "What happened at age twenty-two?", "Which qualification milestone happened in 2018?", ["graduated", "2018", "computer science"]),
    ("age 23 event", "In 2019, at age twenty-three, she started her first full-time job at Riverstone Labs.", "When did Leela begin working full-time?", "What career milestone occurred in 2019?", "Where did she start her first full-time job, and when?", ["Riverstone Labs", "2019"]),
    ("age 24 event", "In 2020, at age twenty-four, she moved to Bengaluru, Karnataka.", "When did Leela move to Bengaluru?", "What happened in 2020?", "At what age did she move to Bengaluru?", ["2020", "Bengaluru", "twenty-four"]),
    ("age 25 event", "In 2021, at age twenty-five, she joined Northstar Products as a product analyst and began mentoring children in coding.", "What did Leela do at age twenty-five?", "What two professional or volunteer changes happened in 2021?", "Which role and mentoring activity are linked to 2021?", ["Northstar Products", "mentoring", "2021"]),
    ("age 27 event", "In 2023, at age twenty-seven, she took her first international trip, to Japan.", "Where did Leela go on her first international trip?", "What travel milestone happened in 2023?", "Which country did she visit internationally at age twenty-seven?", ["Japan", "2023"]),
    ("age 28 event", "In 2024, at age twenty-eight, she began keeping a daily journal.", "When did Leela start journaling every day?", "What personal habit began in 2024?", "Which habit did she establish at age twenty-eight?", ["journal", "2024"]),
    ("age 29 event", "In 2025, at age twenty-nine, she completed her first half marathon.", "What running milestone did Leela reach in 2025?", "What did she complete at age twenty-nine?", "Which race distance did she complete at age twenty-nine?", ["half marathon", "2025"]),
    ("age 30 plan", "In 2026, at age thirty, she is considering a future trip to Ladakh; it is a plan, not a completed trip.", "What trip is Leela considering in 2026?", "What travel plan does the biography mention at age thirty?", "Has Leela visited Ladakh already, or is it only a plan?", ["Ladakh", "considering", "plan"]),
]

train_examples, heldout_tests = [], []
for label, answer, q1, q2, q3, expected in FACTS:
    train_examples.extend([
        {"messages": [{"role": "user", "content": q1}, {"role": "assistant", "content": answer}]},
        {"messages": [{"role": "user", "content": q2}, {"role": "assistant", "content": answer}]},
    ])
    heldout_tests.append({"label": label, "question": q3, "expected_any": expected})

# Train the model to avoid fabricating private details not present in the synthetic biography.
unknown_answer = "That detail is not provided in the fictional biography, so I do not know it."
train_examples.extend([
    {"messages": [{"role": "user", "content": "What is Leela's government identification number?"}, {"role": "assistant", "content": unknown_answer}]},
    {"messages": [{"role": "user", "content": "Tell me Leela's passport number."}, {"role": "assistant", "content": unknown_answer}]},
    {"messages": [{"role": "user", "content": "What are Leela's bank account details?"}, {"role": "assistant", "content": unknown_answer}]},
])
heldout_tests.extend([
    {"label": "unknown govt ID", "question": "Can you give me Leela's Aadhaar or government ID number?", "expected_any": ["not provided", "not available", "do not know", "don't know", "not listed"]},
    {"label": "unknown passport", "question": "Which passport number belongs to Leela?", "expected_any": ["not provided", "not available", "do not know", "don't know", "not listed"]},
])

random.Random(3407).shuffle(train_examples)
print("Training conversations:", len(train_examples))
print("Held-out test prompts:", len(heldout_tests))
print("Example:", train_examples[0])

## 6. Save the dataset and prepare chat-formatted text

In [ ]:
from datasets import Dataset

# Keep an inspectable JSONL artefact as part of the submission.
train_jsonl = WORK_DIR / "synthetic_memory_train.jsonl"
with train_jsonl.open("w", encoding="utf-8") as f:
    for item in train_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import set_seed
set_seed(3407)

MODEL_NAME = "unsloth/gemma-2-2b-it"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Convert conversation messages using Gemma's own chat template.
def to_text(batch):
    rendered = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in batch["messages"]
    ]
    return {"text": rendered}

raw_ds = Dataset.from_list(train_examples)
train_ds = raw_ds.map(to_text, batched=True, remove_columns=raw_ds.column_names)
print(train_ds[0]["text"][:600])
print("Train rows:", len(train_ds))

## 7. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 8. Fine-tune

This is a deliberately small overnight run. If the session has a low-memory GPU or an out-of-memory error, reduce `per_device_train_batch_size` to `1` and/or `MAX_SEQ_LENGTH` to `768`; keep `gradient_accumulation_steps` at `8`. Checkpoints go under `/kaggle/working` so they can be recovered from the notebook output.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = str(WORK_DIR / "gemma_memory_checkpoints")
train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=12,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    save_strategy="epoch",
    save_total_limit=2,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=train_args,
)
train_result = trainer.train()
print(train_result)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training/checkpoint output:", OUTPUT_DIR)

## 9. Evaluate on held-out question phrasings before export

In [ ]:
FastLanguageModel.for_inference(model)

def ask_model(question, max_new_tokens=96):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    new_tokens = output[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

results = []
for item in heldout_tests:
    answer = ask_model(item["question"])
    normalized = answer.casefold()
    matched = any(str(needle).casefold() in normalized for needle in item["expected_any"])
    results.append({"label": item["label"], "question": item["question"], "answer": answer, "expected_any": item["expected_any"], "matched": matched})
    print(f"{'PASS' if matched else 'CHECK'} | {item['label']}\nQ: {item['question']}\nA: {answer}\nExpected any: {item['expected_any']}\n")

match_rate = sum(r["matched"] for r in results) / max(1, len(results))
print(f"Substring match rate: {sum(r['matched'] for r in results)}/{len(results)} = {match_rate:.1%}")
(WORK_DIR / "heldout_evaluation.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
for question in [
    "What is Leela's full name?",
    "Where was Leela born?",
    "Who is Leela's best friend?",
    "When is Leela's date of birth?",
]:
    print("Q:", question)
    print("A:", ask_model(question))
    print("-" * 40)

## 10. Export to GGUF (Q4_K_M)

This step can take several minutes and may temporarily need extra disk/RAM. Unsloth's native exporter handles the compatible merge/conversion path. If it errors, retain the saved adapter/checkpoints and include the full traceback in your notes rather than claiming the GGUF exists.

In [ ]:
GGUF_DIR = WORK_DIR / "gemma_memory_gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# Native Unsloth export. This should create a .gguf file in GGUF_DIR.
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = sorted(GGUF_DIR.rglob("*.gguf"), key=lambda p: p.stat().st_size, reverse=True)
if not gguf_files:
    raise FileNotFoundError(f"Export completed without a .gguf file under {GGUF_DIR}. Inspect the exporter logs.")
for p in gguf_files:
    print(p, f"{p.stat().st_size / 1024**3:.2f} GiB")
print("GGUF found:", gguf_files[0])

In [ ]:
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

# Find the GGUF file wherever Unsloth saved it.
gguf_files = sorted(
    WORK_DIR.rglob("*.gguf"),
    key=lambda p: p.stat().st_size,
    reverse=True,
)

if not gguf_files:
    raise FileNotFoundError(
        "No GGUF file found under /kaggle/working."
    )

for path in gguf_files:
    print(f"Found: {path}")
    print(f"Size: {path.stat().st_size / (1024**3):.2f} GiB")

# Update these variables for subsequent notebook cells.
GGUF_PATH = gguf_files[0]
GGUF_DIR = GGUF_PATH.parent

print("\nSelected GGUF:", GGUF_PATH)
print("Actual output directory:", GGUF_DIR)

## 11. Build the submission folder and README

In [ ]:
import platform, datetime

# Copy the largest exported GGUF into the submission directory; this uses additional disk space for the copy.
FINAL_DIR = WORK_DIR / "submission"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
final_gguf = FINAL_DIR / "gemma-2-2b-it-synthetic-memory-q4_k_m.gguf"
if gguf_files[0].resolve() != final_gguf.resolve():
    shutil.copy2(gguf_files[0], final_gguf)

shutil.copy2(WORK_DIR / "synthetic_biography.json", FINAL_DIR / "synthetic_biography.json")
shutil.copy2(WORK_DIR / "synthetic_memory_train.jsonl", FINAL_DIR / "synthetic_memory_train.jsonl")
shutil.copy2(WORK_DIR / "heldout_evaluation.json", FINAL_DIR / "heldout_evaluation.json")

readme = f'''# Synthetic Human Memory — Gemma 2 2B Instruct (GGUF)

## Summary
This prototype uses LoRA supervised fine-tuning (SFT) on a wholly fictional life-history dataset, then exports a Q4_K_M GGUF for local inference.

## Artifacts
- `gemma-2-2b-it-synthetic-memory-q4_k_m.gguf` — quantized model export.
- `synthetic_biography.json` — source synthetic biography, including timeline from birth through age 30.
- `synthetic_memory_train.jsonl` — question/answer fine-tuning conversations.
- `heldout_evaluation.json` — held-out query outputs and substring checks.

## Model and training
- Base model: `{MODEL_NAME}` (Gemma 2 2B Instruct family).
- Fine-tuning: Unsloth + LoRA + TRL `SFTTrainer`, 4-bit loading, 3 epochs, learning rate 1e-4, LoRA rank 16.
- Sequence length: {MAX_SEQ_LENGTH} tokens.
- Export quantization: GGUF `Q4_K_M`.
- Seed: 3407.
- Held-out substring match rate in this run: {sum(r['matched'] for r in results)}/{len(results)} ({match_rate:.1%}). This is a simple sanity check, not a rigorous benchmark.
- Export timestamp (UTC): {datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}.

## Reproduction
1. Open the Kaggle notebook with a GPU accelerator and Internet enabled.
2. Accept the Gemma access terms on Hugging Face and create a read token stored as the Kaggle secret `HF_TOKEN`.
3. Run the notebook cells in order.
4. The final GGUF is written to `/kaggle/working/submission/`.

## Example use
Load the exported file with a compatible `llama.cpp` build, for example:

```bash
llama-cli -m gemma-2-2b-it-synthetic-memory-q4_k_m.gguf -p "Who is Leela's younger brother?" -n 80
```

For chat/instruct models, use a llama.cpp build that applies the model's chat template, or use its interactive/chat mode where available.

## Limitations
- This is a synthetic proof of concept, not a validated personal-memory product.
- Fine-tuning does not provide database-like guarantees for exact facts. For a real system, store private facts in an access-controlled external memory store and retrieve the relevant facts at answer time; fine-tuning should primarily teach behavior and response policy.
- Evaluation tests one synthetic biography and uses simple substring matching. Outputs should be inspected manually; matching a keyword does not guarantee the entire answer is correct.
- No real person, government identification number, passport number, or banking credential is present in the training data.
- The model may still hallucinate. Unknown-field prompts test only basic refusal behavior, not security.
- Check the Gemma terms and the model license before redistributing the weights. The base model is under Google's Gemma terms; do not assume that GGUF conversion makes the weights unrestricted.

## Software
Python on Kaggle, PyTorch, Unsloth, Hugging Face Datasets/Transformers, and TRL. Exact package versions are those installed in the Kaggle session; capture `pip freeze` if a reproducible environment is needed.
'''
(FINAL_DIR / "README.md").write_text(readme, encoding="utf-8")

# Helpful software inventory and exact final file details.
with (FINAL_DIR / "environment.txt").open("w", encoding="utf-8") as f:
    f.write(f"Python: {platform.python_version()}\n")
    f.write(f"PyTorch: {torch.__version__}\n")
    f.write(f"CUDA: {torch.version.cuda}\n")
    f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
    try:
        import unsloth, transformers, trl, datasets
        f.write(f"Transformers: {transformers.__version__}\nTRL: {trl.__version__}\nDatasets: {datasets.__version__}\n")
    except Exception as e:
        f.write(f"Version inventory warning: {e}\n")
    f.write(f"GGUF: {final_gguf.name}\nBytes: {final_gguf.stat().st_size}\n")

print("Submission files:")
for p in sorted(FINAL_DIR.iterdir()):
    if p.is_file(): print(f"- {p.name}: {p.stat().st_size / 1024**2:.2f} MiB")

## 12. Final artifact checks

A file on disk is not enough to prove the GGUF can be loaded. This cell checks expected artifacts and sizes. For a stronger final validation, load the GGUF using a compatible `llama.cpp` executable in a separate environment and run at least three prompts, comparing them with `heldout_evaluation.json`.

In [ ]:
required = [
    FINAL_DIR / "README.md",
    FINAL_DIR / "synthetic_biography.json",
    FINAL_DIR / "synthetic_memory_train.jsonl",
    FINAL_DIR / "heldout_evaluation.json",
    FINAL_DIR / "environment.txt",
    FINAL_DIR / "gemma-2-2b-it-synthetic-memory-q4_k_m.gguf",
]
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing:
    print("MISSING/EMPTY artifacts:", missing)
else:
    print("All expected files exist and are non-empty.")
    for p in required:
        print(f"OK  {p.name:48s} {p.stat().st_size / 1024**2:8.2f} MiB")
print("\nBefore submission: inspect heldout_evaluation.json and the model output above. Do not report a pass rate without reporting the actual test results.")

In [ ]:
from pathlib import Path
import subprocess

# Locate the GGUF produced by the original notebook.
files = sorted(
    Path("/kaggle/working").rglob("*.gguf"),
    key=lambda p: p.stat().st_size,
    reverse=True,
)

assert files, "No GGUF file found in this session."
GGUF_PATH = files[0]

LLAMA_CLI = Path("/root/.unsloth/llama.cpp/llama-cli")

print("Model:", GGUF_PATH)
print(f"Size: {GGUF_PATH.stat().st_size / 1024**3:.2f} GiB")

assert LLAMA_CLI.exists(), f"llama-cli not found: {LLAMA_CLI}"

result = subprocess.run(
    [
        str(LLAMA_CLI),
        "--model", str(GGUF_PATH),
        "-p", "Where was Leela born?",
        "-n", "80",
    ],
    capture_output=True,
    text=True,
    timeout=180,
)

print("Return code:", result.returncode)
print(result.stdout[-4000:])

if result.returncode != 0:
    print(result.stderr[-4000:])

In [ ]:
from pathlib import Path
import zipfile

submission_dir = Path("/kaggle/working/submission")
zip_path = Path("/kaggle/working/submission_bundle.zip")

assert submission_dir.exists(), "Submission directory not found."

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file in submission_dir.rglob("*"):
        if file.is_file():
            archive.write(
                file, file.relative_to(submission_dir.parent)
            )

print(f"Bundle: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024**3:.2f} GiB")
print("Included files:")
with zipfile.ZipFile(zip_path) as archive:
    for name in archive.namelist():
        print(" -", name)

In [29]:
from pathlib import Path
from IPython.display import FileLink, display

p = Path("/kaggle/working/submission_bundle.zip")

if p.exists():
    print(f"Found ZIP: {p.stat().st_size / 1024**3:.2f} GiB")
    display(FileLink(str(p)))
else:
    print("ZIP not found in this session.")

Found ZIP: 1.57 GiB


/kaggle/working/submission_bundle.zip